<a href="https://colab.research.google.com/github/TivyaIshwarya/ai-support-escalation-agent/blob/main/RAG_with_Llamaindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install sentence-transformers faiss-cpu flask pyngrok nest-asyncio

In [5]:
import json

tickets = [
    {"question": "How to reset password?",
     "answer": "Click on forgot password and follow the instructions."},

    {"question": "Payment failed but money deducted",
     "answer": "Wait 24 hours. If not refunded, contact support with transaction ID."},

    {"question": "App is crashing after update",
     "answer": "Clear cache and reinstall the app."}
]

with open("tickets.json", "w") as f:
    json.dump(tickets, f)

In [6]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

class RAGEngine:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.load_data()

    def load_data(self):
        with open("tickets.json", "r") as f:
            data = json.load(f)

        self.documents = data
        texts = [item["question"] for item in data]
        embeddings = self.model.encode(texts)

        self.dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(np.array(embeddings))

    def retrieve(self, query):
        query_embedding = self.model.encode([query])
        distances, indices = self.index.search(np.array(query_embedding), k=1)
        return self.documents[indices[0][0]], distances[0][0]

rag = RAGEngine()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
def should_escalate(distance, threshold=1.2):
    return distance > threshold

In [8]:
def ask_support(query):
    result, distance = rag.retrieve(query)

    if should_escalate(distance):
        return "Escalated to human agent 🚨"

    return result["answer"]

print(ask_support("I forgot my password"))

Click on forgot password and follow the instructions.


In [10]:
print(ask_support("How to reset password?"))
print(ask_support("Money deducted but payment failed"))
print(ask_support("My account hacked by someone"))

Click on forgot password and follow the instructions.
Wait 24 hours. If not refunded, contact support with transaction ID.
Click on forgot password and follow the instructions.
